In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# NLP 평가지표 라이브러리 로드
import evaluate
from bert_score import score as clac_bert_score

In [2]:
# =====================================================================
# [설정 영역] 파일 경로를 지정하세요.
# =====================================================================
CSV_PATH = "vlm_results_v1.csv"         # 입력 파일 경로
OUTPUT_CSV_PATH = "vlm_evaluation_results.csv" # 점수가 추가되어 저장될 파일 경로
# =====================================================================

In [3]:
print("🔄 평가 모델 및 메트릭 로딩 중...")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# 1. 데이터셋 로드 및 전처리
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"❌ {CSV_PATH} 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")

df = pd.read_csv(CSV_PATH)

# 결측치나 빈 문자열 방어 처리 (String 변환)
df['findings'] = df['findings'].fillna("").astype(str).str.strip()
df['vlm_output'] = df['vlm_output'].fillna("").astype(str).str.strip()

# 데이터 개수 확인
total_rows = len(df)
print(f"📄 총 {total_rows}개의 데이터 쌍을 분석합니다.")

# 점수를 담을 리스트 초기화
bleu_1_list, bleu_4_list, rouge_l_list = [], [], []

print("\n🧮 1단계: BLEU 및 ROUGE 점수 고속 계산 중...")
for idx, row in tqdm(df.iterrows(), total=total_rows):
    ref = row['findings']
    pred = row['vlm_output']
    
    # 둘 중 하나라도 비어있으면 최하점 처리
    if not ref or not pred:
        bleu_1_list.append(0.0)
        bleu_4_list.append(0.0)
        rouge_l_list.append(0.0)
        continue
        
    try:
        # BLEU 계산 (단어 매칭 개수별 가중치 조절)
        # max_order=1은 단어 1개 매칭(BLEU-1), max_order=4는 연속된 4개 단어 매칭(BLEU-4)
        b1 = bleu_metric.compute(predictions=[pred], references=[[ref]], max_order=1)['bleu']
        b4 = bleu_metric.compute(predictions=[pred], references=[[ref]], max_order=4)['bleu']
        
        # ROUGE-L 계산 (최장 공통 부분 순열 기반)
        r_l = rouge_metric.compute(predictions=[pred], references=[ref])['rougeL']
        
        bleu_1_list.append(b1)
        bleu_4_list.append(b4)
        rouge_l_list.append(r_l)
    except Exception as e:
        bleu_1_list.append(0.0)
        bleu_4_list.append(0.0)
        rouge_l_list.append(0.0)

# 결과 데이터프레임에 이식
df['bleu_1'] = bleu_1_list
df['bleu_4'] = bleu_4_list
df['rouge_l'] = rouge_l_list


print("\n🧠 2단계: BERTScore 딥러닝 문맥 유사도 계산 중 (GPU 가속 활용)...")
# 리스트 형태로 한 번에 던져서 GPU 병렬 연산을 수행합니다.
preds_list = df['vlm_output'].tolist()
refs_list = df['findings'].tolist()

# lang="en" 지정 시 기본적으로 roberta-large 모델을 다운로드하여 연산합니다.
P, R, F1 = clac_bert_score(preds_list, refs_list, lang="en", verbose=False)
df['bert_score_f1'] = F1.numpy()


# 3. 최종 파일 저장 및 스코어보드 출력
df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
print(f"\n💾 각 행별 점수 매핑 완료! 파일이 저장되었습니다 ➡️ {OUTPUT_CSV_PATH}")

print("\n" + "="*50)
print("📊 [최종 종합 성능 결과 보고서] (Dataset Average)")
print("="*50)
print(f"🔹 BLEU-1 (단어 일치율)      : {df['bleu_1'].mean() * 100:.2f} 점")
print(f"🔹 BLEU-4 (문장 구조 일치율)  : {df['bleu_4'].mean() * 100:.2f} 점")
print(f"🔹 ROUGE-L (핵심 소견 재현율) : {df['rouge_l'].mean() * 100:.2f} 점")
print(f"🔹 BERTScore F1 (의미론적 유사도): {df['bert_score_f1'].mean() * 100:.2f} 점")
print("="*50)
print("💡 TIP: 아까처럼 VLM이 안전빵 정상 문장으로 대거 밀어버린 상태라면,\n"
      "        의미가 완전히 엇갈려서 BLEU-4 점수가 현저히 낮게 출력될 것입니다.")

🔄 평가 모델 및 메트릭 로딩 중...
📄 총 800개의 데이터 쌍을 분석합니다.

🧮 1단계: BLEU 및 ROUGE 점수 고속 계산 중...


100%|██████████| 800/800 [00:29<00:00, 26.73it/s]



🧠 2단계: BERTScore 딥러닝 문맥 유사도 계산 중 (GPU 가속 활용)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/minkyujeong/anaconda3/envs/mini_thon/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Dow


💾 각 행별 점수 매핑 완료! 파일이 저장되었습니다 ➡️ vlm_evaluation_results.csv

📊 [최종 종합 성능 결과 보고서] (Dataset Average)
🔹 BLEU-1 (단어 일치율)      : 31.10 점
🔹 BLEU-4 (문장 구조 일치율)  : 5.39 점
🔹 ROUGE-L (핵심 소견 재현율) : 23.38 점
🔹 BERTScore F1 (의미론적 유사도): 87.92 점
💡 TIP: 아까처럼 VLM이 안전빵 정상 문장으로 대거 밀어버린 상태라면,
        의미가 완전히 엇갈려서 BLEU-4 점수가 현저히 낮게 출력될 것입니다.
